# 03 Validation

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os
os.chdir("/content/drive/MyDrive/Gates Foundation/Building Dataset Validation/")

Mounted at /content/drive


In [2]:
from pathlib import Path

PROJECT_ROOT = Path("/content/drive/MyDrive/Gates Foundation/Building Dataset Validation")
CONFIG_PATH  = PROJECT_ROOT / "configs/validation_configs.yaml"

# Set overwrite=True to re-run cities whose outputs already exist.
# When False (default), cities with existing sentinel parquets are skipped automatically.
OVERWRITE = True

In [3]:
import sys, os
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

In [4]:
import logging
import yaml
from src.validator import UrbanValidator

for handler in logging.root.handlers[:]:
    logging.root.removeHandler(handler)

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s  %(levelname)-8s  %(message)s",
    datefmt="%H:%M:%S",
    force=True,
)

# Patch overwrite flag into config before constructing Validator
with open(CONFIG_PATH) as f:
    _cfg_preview = yaml.safe_load(f)

datasets_preview = _cfg_preview.get("vector", {}).get("datasets", [])
enabled = [d["name"] for d in datasets_preview if d.get("enabled", True)]
print(f"Config: {CONFIG_PATH}")
print(f"Enabled candidate datasets: {enabled}")

Config: /content/drive/MyDrive/Gates Foundation/Building Dataset Validation/configs/validation_configs.yaml
Enabled candidate datasets: ['overture', 'gba', 'globfp']


In [5]:
# Instantiate the validator — this reads the config and AOI tracker,
# resolves file paths, and logs how many datasets are queued.
# The overwrite flag from the cell above is injected before loading.
import yaml, copy

with open(CONFIG_PATH) as f:
    _cfg_patched = yaml.safe_load(f)
_cfg_patched.setdefault("output", {})["overwrite"] = OVERWRITE

# Write patched config to a temp file so Validator can read it
import tempfile, os
_tmp = tempfile.NamedTemporaryFile(
    mode="w", suffix=".yaml", delete=False, dir=PROJECT_ROOT / "configs"
)
yaml.dump(_cfg_patched, _tmp)
_tmp.close()
_PATCHED_CONFIG_PATH = _tmp.name

v = UrbanValidator(_PATCHED_CONFIG_PATH)
print(f"\nDatasets queued: {len(v.datasets)}")
for ds in v.datasets:
    print(f"  {ds['id']}")

13:05:23  INFO      Validation tracker: 85 -> 85 suitable rows.
13:06:36  INFO      Loaded 85 dataset(s) for validation.
13:06:36  INFO      Loaded 85 dataset(s) for validation.



Datasets queued: 85
  afg-ayabak
  afg-bamyan
  afg-bazarak
  afg-charikar
  afg-east-kabul
  afg-ferozkoh
  afg-gardez
  afg-herat
  afg-jalalaba
  afg-mahmud-e-raqi
  afg-mazar-e-sharif
  afg-poruns
  afg-qala-e-naw
  afg-west-kabul
  afg-zaranj
  ant-curacao
  bgd-rohingya
  blz-burrell-boom
  bra-nova-sussuarana
  col-san-antonio-de-prado
  cvg-saint-vincent-grenadines
  dom-dominica
  gha-accra
  gha-aiyim-sraha
  gha-dansoman
  gha-nawuni
  gha-sawla-tuna
  gha-wa
  grd-grenada
  jam-kingston
  jam-saint-catherine
  jpn-ashiya-hama
  jpn-hiroshima
  jpn-iwate
  jpn-izu-oshima
  jpn-kashima
  jpn-numakunai
  ken-kakuma
  ken-kakuma-kalobeyei
  ken-mukuru
  ken-nairobi
  lbr-monrovia
  lby-almarj
  lby-bayda
  lby-darnah
  lby-susah
  lca-saint-lucia
  maf-saint-martin
  mmr-patheingyi-mandalay
  moz-de-maio
  moz-djonasse
  mwi-lilongwe
  mwi-mlowe
  ner-niame
  nga-ibadan
  phl-bagamanoc
  phl-barangay
  phl-catanduanes
  phl-juraojurao-anini-y-antique
  phl-pasig
  phl-poblacio

In [6]:
import pandas as pd

results = v.validate_vector()

# Clean up temp config
try:
    os.unlink(_PATCHED_CONFIG_PATH)
except Exception:
    pass

# Summary
summary = pd.DataFrame(
    [{"aoi": k, "status": "ok" if v else "failed"} for k, v in results.items()]
)
print(f"\nDone — {len(summary)} AOIs processed.\n")
display(summary.groupby("status")["aoi"].count().rename("count").to_frame())
display(summary)

13:06:38  INFO      ━━━━  afg-ayabak  ━━━━
13:06:38  INFO      MEM [afg-ayabak start] RSS = 354 MB
13:06:38  INFO      [afg-ayabak] Working CRS: EPSG:32642 | 51 tiles.
13:06:38  INFO      Created 51 records
13:06:41  INFO      [afg-ayabak] Reference buildings: 18702 (from 1 file(s))
13:06:41  INFO      [afg-ayabak / overture] Candidate: afg_ayabak_overture_building.parquet
13:06:42  INFO      [afg-ayabak / overture] Candidate buildings: 25507
13:07:19  INFO      [afg-ayabak / overture] Tile metrics saved → vector_metrics_tiles_overture.parquet
13:07:20  INFO      MEM [afg-ayabak/overture done] RSS = 548 MB
13:07:20  INFO      [afg-ayabak / gba] Candidate: afg_ayabak_gba.parquet
13:07:21  INFO      [afg-ayabak / gba] Candidate buildings: 25682
13:07:55  INFO      [afg-ayabak / gba] Tile metrics saved → vector_metrics_tiles_gba.parquet
13:07:56  INFO      MEM [afg-ayabak/gba done] RSS = 568 MB
13:07:56  INFO      [afg-ayabak / globfp] Candidate: afg_ayabak_globfp.parquet
13:07:57  INFO  

[afg_east_kabul_gba.parquet] fixing 2 invalid geometries...


13:22:19  INFO      [afg-east-kabul / gba] Candidate buildings: 290299
13:31:24  INFO      [afg-east-kabul / gba] Tile metrics saved → vector_metrics_tiles_gba.parquet
13:31:25  INFO      MEM [afg-east-kabul/gba done] RSS = 1662 MB
13:31:25  INFO      [afg-east-kabul / globfp] Candidate: afg_east_kabul_globfp.parquet


[afg_east_kabul_globfp.parquet] fixing 11 invalid geometries...


13:31:30  INFO      [afg-east-kabul / globfp] Candidate buildings: 185094
13:36:39  INFO      [afg-east-kabul / globfp] Tile metrics saved → vector_metrics_tiles_globfp.parquet
13:36:40  INFO      MEM [afg-east-kabul/globfp done] RSS = 1648 MB
13:36:43  INFO      [afg-east-kabul] City summary saved.
13:36:48  INFO      MEM [afg-east-kabul end] RSS = 1508 MB
13:36:48  INFO      [afg-east-kabul] ✓ Complete.
13:36:48  INFO      ━━━━  afg-ferozkoh  ━━━━
13:36:48  INFO      MEM [afg-ferozkoh start] RSS = 1508 MB
13:36:48  INFO      [afg-ferozkoh] Working CRS: EPSG:32641 | 34 tiles.
13:36:48  INFO      Created 34 records
13:36:50  INFO      [afg-ferozkoh] Reference buildings: 9839 (from 1 file(s))
13:36:50  INFO      [afg-ferozkoh / overture] Candidate: afg_ferozkoh_overture_building.parquet
13:36:51  INFO      [afg-ferozkoh / overture] Candidate buildings: 13025
13:37:07  INFO      [afg-ferozkoh / overture] Tile metrics saved → vector_metrics_tiles_overture.parquet
13:37:07  INFO      MEM [

...

[afg_jalalaba_globfp.parquet] fixing 1 invalid geometries...


13:59:34  INFO      [afg-jalalaba / globfp] Candidate buildings: 81168
14:01:13  INFO      [afg-jalalaba / globfp] Tile metrics saved → vector_metrics_tiles_globfp.parquet
14:01:13  INFO      MEM [afg-jalalaba/globfp done] RSS = 1621 MB
14:01:15  INFO      [afg-jalalaba] City summary saved.
14:01:19  INFO      MEM [afg-jalalaba end] RSS = 1583 MB
14:01:19  INFO      [afg-jalalaba] ✓ Complete.
14:01:19  INFO      ━━━━  afg-mahmud-e-raqi  ━━━━
14:01:19  INFO      MEM [afg-mahmud-e-raqi start] RSS = 1583 MB
14:01:19  INFO      [afg-mahmud-e-raqi] Working CRS: EPSG:32642 | 51 tiles.
14:01:20  INFO      Created 51 records
14:01:21  INFO      [afg-mahmud-e-raqi] Reference buildings: 10655 (from 1 file(s))
14:01:21  INFO      [afg-mahmud-e-raqi / overture] Candidate: afg_mahmud_e_raqi_overture_building.parquet
14:01:23  INFO      [afg-mahmud-e-raqi / overture] Candidate buildings: 13418
14:01:41  INFO      [afg-mahmud-e-raqi / overture] Tile metrics saved → vector_metrics_tiles_overture.parqu

[mazar-e-sharif_WGB.geojson] fixing 1 invalid geometries...


14:02:28  INFO      [afg-mazar-e-sharif] Reference buildings: 127820 (from 1 file(s))
14:02:28  INFO      [afg-mazar-e-sharif / overture] Candidate: afg_mazar_e_sharif_overture_building.parquet
14:02:35  INFO      [afg-mazar-e-sharif / overture] Candidate buildings: 134793
14:04:07  ERROR     [afg-mazar-e-sharif] Unhandled error during vector validation.
Traceback (most recent call last):
  File "/content/drive/MyDrive/Gates Foundation/Building Dataset Validation/src/validator.py", line 80, in validate_vector
    results[ds["id"]] = self._validate_vector_dataset(ds)
                        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/content/drive/MyDrive/Gates Foundation/Building Dataset Validation/src/validator.py", line 192, in _validate_vector_dataset
    tile_path, match_path = self._run_candidate(
                            ^^^^^^^^^^^^^^^^^^^^
  File "/content/drive/MyDrive/Gates Foundation/Building Dataset Validation/src/validator.py", line 307, in _run_candidate
    metrics, ma

: 

In [ ]:
summary.groupby("status")["aoi"].count().rename("count").to_frame()

In [ ]:
summary

,aoi,status
0,ant-curacao,ok
1,bgd-rohingya,ok
2,blz-burrell-boom,ok
3,bra-nova-sussuarana,ok
4,col-san-antonio-de-prado,ok
...,...,...
65,uga-bugoye,ok
66,uga-kampala,ok
67,uga-kanara,ok
68,uga-nakamiro,ok
